In [ ]:
import requests
import pandas as pd

url = "https://charts.youtube.com/youtubei/v1/browse?alt=json"

headers = {
    "Content-Type": "application/json"
}

payload = {
    "context": {
        "client": {
            "clientName": "WEB_MUSIC_ANALYTICS",
            "clientVersion": "2.0",
            "hl": "en-GB",
            "gl": "IN",
            "theme": "MUSIC"
        }
    },
    "browseId": "FEmusic_analytics_charts_home",
    "query": (
        "perspective=CHART_DETAILS"
        "&chart_params_country_code=in"
        "&chart_params_chart_type=TRACKS"
        "&chart_params_period_type=WEEKLY"
        "&chart_params_end_date=20250710"
    )
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)

data = response.json()


# ----------------------------------------------------
# FIND trackViews OBJECT
# ----------------------------------------------------

def find_trackviews(obj):

    if isinstance(obj, dict):

        if "trackViews" in obj:
            return obj["trackViews"]

        for value in obj.values():
            result = find_trackviews(value)

            if result:
                return result

    elif isinstance(obj, list):

        for item in obj:
            result = find_trackviews(item)

            if result:
                return result

    return None


tracks = find_trackviews(data)

print(f"Found {len(tracks)} tracks")


# ----------------------------------------------------
# CLEAN EXTRACTION
# ----------------------------------------------------

rows = []

for track in tracks:

    artists = ", ".join(
        [a["name"] for a in track.get("artists", [])]
    )

    rank = (
        track.get("chartEntryMetadata", {})
        .get("currentPosition")
    )

    rows.append({
        "rank": rank,
        "song": track.get("name"),
        "artists": artists,
        "views": int(track.get("viewCount", 0)),
        "video_id": track.get("encryptedVideoId"),
        "weeks_on_chart": (
            track.get("chartEntryMetadata", {})
            .get("periodsOnChart")
        )
    })


df = pd.DataFrame(rows)

df = df.sort_values("rank")

print(df.head(20))


# ----------------------------------------------------
# SAVE
# ----------------------------------------------------

df.to_csv("india_weekly_chart.csv", index=False)

print("\nSaved to india_weekly_chart.csv")